# Заняття 12 — Spark: Структуровані дані. Частина 1

## Основні цілі заняття:
* Познайомитися з двома основними інтерфейсами Spark: **DataFrame API** і **SparkSQL**.
* Зрозуміти різницю між **transformations** і **actions**.
* Освоїти базові операції: select, filter, groupBy, join, window functions.
* Побачити RDD як низькорівневий API — контекст для розуміння архітектури.
* Провести перші реальні обчислення над NYC Taxi Dataset у Spark.

* **Датасет:** NYC TLC Yellow Taxi Trip Records, January 2024 — той самий файл, що в заняттях 02 і 03
* **Формат:** Parquet (~100 MB, ~3 млн рядків)
* **Довідник зон:** NYC TLC Taxi Zone Lookup (265 рядків)
* **Spark:** local mode, PySpark 4.1.1 (`uv run jupyter lab`)

Структура ноутбука:
1. Ініціалізація SparkSession
2. Читання даних — Parquet; schema inference vs explicit schema
3. RDD — низькорівневий API
4. DataFrame API: select, filter, withColumn, orderBy, alias, drop
5. Агрегації: groupBy, agg, rollup, describe
6. Joins: inner, left, cross + explain()
7. Union, intersect, exceptAll
8. Null-handling: fillna, dropna, isNull / isNotNull
9. SparkSQL: createOrReplaceTempView, spark.sql, selectExpr
10. Window functions: row_number, rank, dense_rank, lag
11. Pandas API on Spark: pyspark.pandas, groupby, agg, to_spark
12. Кешування та управління пам'яттю: cache, persist, StorageLevel, unpersist
13. Repartition і Coalesce: getNumPartitions, repartition, coalesce, hash partitioning
14. Запис даних: write, partitionBy, вплив партиціонування на кількість файлів

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, TimestampType,
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from icecream import ic

PARQUET_PATH = "data/landing/yellow_tripdata_2024-01.parquet"
ZONE_PARQUET = "data/reference/taxi_zone_lookup.parquet"

## 1. Ініціалізація SparkSession

`SparkSession` — єдина точка входу до Spark у PySpark 3.x.
У local mode всі executor'и запускаються всередині одного JVM-процесу.
`local[*]` — використати всі доступні ядра.

Порівняння із заняттям 02: `pd.read_parquet(...)` вантажить усі дані в один процес.
Spark ділить дані на **partitions** і обробляє їх паралельно.

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("lesson-12-nyc-taxi")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.execution.arrow.pyspark.enabled", True)
    .getOrCreate()
)

ic(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/13 18:40:00 WARN Utils: Your hostname, MacBook-Air-Illia.local, resolves to a loopback address: 127.0.0.1; using 192.168.50.236 instead (on interface en0)
26/08/13 18:40:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 18:40:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
ic| spark.version: '4.1.1'


'4.1.1'

## 2. Читання даних

Spark — **schema-on-read**: схема виводиться з Parquet footer без сканування рядків.
Той самий підхід, що і `pq.read_schema()` у занятті 02.

### Parquet — schema inference

In [3]:
df = spark.read.parquet(PARQUET_PATH)

In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
df

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0


`explain()` — фізичний план виконання. Для простого читання — один вузол `FileScan Parquet`.

In [6]:
df.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: parquet, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double
Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] parquet

== Optimized Logical Plan ==
Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_

### Explicit schema з StructType

Явна схема гарантує типи незалежно від вмісту файлу.
Spark пропускає footer scan → скорочує time-to-first-result для великих наборів файлів.

In [7]:
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", IntegerType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
])

df_typed = spark.read.schema(schema).parquet(PARQUET_PATH)
df_typed.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



### Інші джерела (reference code)

Spark читає CSV, JSON, JDBC, Kafka через той самий `spark.read` API.
Приклади нижче потребують відповідних коннекторів або файлів.

In [8]:
# CSV
# df_csv = spark.read.options(header=True, inferSchema=True, delimiter=",").csv("path/to/file.csv")

# JSON
# df_json = spark.read.json("path/to/file.json", multiLine=True)

In [9]:
# JDBC з partitioning
# df_jdbc = spark.read.jdbc(
#     url="jdbc:postgresql://host:5432/db",
#     table="trips",
#     properties={"user": "...", "password": "...", "driver": "org.postgresql.Driver"},
#     column="id", lowerBound=1, upperBound=100000, numPartitions=10,
# )

## 3. RDD — низькорівневий API

DataFrame API перекладає операції в план запиту, який Spark оптимізує через **Catalyst**.
RDD — рівень нижче: явні `map`, `filter`, `reduceByKey` без оптимізатора.
Розуміння RDD потрібне для читання старого коду і розуміння того, що відбувається
«під капотом» DataFrames.

In [10]:
sc = spark.sparkContext

# sc.parallelize — створити RDD з Python-колекції (для демонстрацій і тестів)
sample_log = [
    "INFO SparkContext: Running Spark version 3.5.3",
    "INFO DAGScheduler: Job 0 finished: count at NativeMethodAccessorImpl",
    "WARN TaskSetManager: Lost task 0.0 in stage 0.0",
    "ERROR SparkContext: Error initializing SparkContext",
    "INFO DAGScheduler: Job 1 finished: show at NativeMethodAccessorImpl",
    "WARN BlockManager: Block rdd_0_0 not found on executor",
]
rdd = sc.parallelize(sample_log)

In [11]:
rdd.take(3)

['INFO SparkContext: Running Spark version 3.5.3',
 'INFO DAGScheduler: Job 0 finished: count at NativeMethodAccessorImpl',
 'WARN TaskSetManager: Lost task 0.0 in stage 0.0']

In [12]:
ic(rdd.count())

ic| rdd.count(): 6


6

In [13]:
ic(rdd.first())

ic| rdd.first(): 'INFO SparkContext: Running Spark version 3.5.3'


'INFO SparkContext: Running Spark version 3.5.3'

### Transformations: filter, map, flatMap

**Transformations** — ліниві: повертають новий RDD, не запускають обчислення.
**Actions** — запускають обчислення і повертають результат (`take`, `count`, `collect`).

In [14]:
warn_rdd = rdd.filter(lambda line: "WARN" in line or "ERROR" in line)
warn_rdd.collect()

['WARN TaskSetManager: Lost task 0.0 in stage 0.0',
 'ERROR SparkContext: Error initializing SparkContext',
 'WARN BlockManager: Block rdd_0_0 not found on executor']

In [15]:
lengths = rdd.map(lambda line: len(line))
lengths.take(5)

[46, 68, 47, 51, 67]

In [16]:
ic(lengths.reduce(lambda a, b: a + b))

ic| lengths.reduce(lambda a, b: a + b): 333


333

In [17]:
words = rdd.flatMap(lambda line: line.split(" "))
words.take(10)

['INFO',
 'SparkContext:',
 'Running',
 'Spark',
 'version',
 '3.5.3',
 'INFO',
 'DAGScheduler:',
 'Job',
 '0']

### Word count — канонічний приклад RDD

In [18]:
word_pairs = words.map(lambda word: (word, 1))
word_counts = word_pairs.reduceByKey(lambda a, b: a + b)

word_counts.sortBy(lambda x: x[1], ascending=False).take(10)

[('INFO', 3),
 ('DAGScheduler:', 2),
 ('SparkContext:', 2),
 ('WARN', 2),
 ('Job', 2),
 ('finished:', 2),
 ('0.0', 2),
 ('at', 2),
 ('NativeMethodAccessorImpl', 2),
 ('task', 1)]

In [19]:
ic(word_counts.distinct().count())

ic| word_counts.distinct().count(): 33


33

### RDD → DataFrame

`spark.createDataFrame(rdd, schema)` — перетворення для подальшої роботи через DataFrame API.

In [20]:
word_schema = StructType([
    StructField("word", StringType(), True),
    StructField("count", IntegerType(), True),
])
spark.createDataFrame(word_counts, word_schema).orderBy(F.col("count").desc()).show(10)

+--------------------+-----+
|                word|count|
+--------------------+-----+
|                INFO|    3|
|                 Job|    2|
|       DAGScheduler:|    2|
|           finished:|    2|
|                 0.0|    2|
|       SparkContext:|    2|
|                  at|    2|
|                WARN|    2|
|NativeMethodAcces...|    2|
|               Spark|    1|
+--------------------+-----+
only showing top 10 rows


### RDD set operations

In [21]:
rdd1 = sc.parallelize([("EWR", 1), ("JFK", 2)])
rdd2 = sc.parallelize([("EWR", 3), ("JFK", 4), ("LGA", 5)])
rdd1.join(rdd2).collect()

[('EWR', (1, 3)), ('JFK', (2, 4))]

In [22]:
rdd_a = sc.parallelize([1, 2, 3])
rdd_b = sc.parallelize([3, 4, 5])
rdd_a.union(rdd_b).collect()

[1, 2, 3, 3, 4, 5]

### DataFrame → RDD

Рідкісний сценарій — коли операція не доступна у DataFrame API.

In [23]:
df.rdd.take(3)

[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 0, 9, 36), passenger_count=1, trip_distance=1.8, RatecodeID=1, store_and_fwd_flag='N', PULocationID=140, DOLocationID=236, payment_type=1, fare_amount=10.0, extra=3.5, mta_tax=0.5, tip_amount=3.75, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=18.75, congestion_surcharge=2.5, Airport_fee=0.0),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 17, 6), tpep_dropoff_datetime=datetime.datetime(2024, 1, 

## 4. DataFrame API — базові операції

Той самий набір операцій, що у Pandas і Polars — але виконується розподілено.

### select

In [24]:
df.select("PULocationID", "DOLocationID", "fare_amount")

PULocationID,DOLocationID,fare_amount
186,79,17.7
140,236,10.0
236,79,23.3
79,211,10.0
211,148,7.9
148,141,29.6
138,181,45.7
246,231,25.4
161,261,31.0
113,113,3.0


In [25]:
# Три рівноцінні способи — рядок, атрибут, F.col()
df.select(df["fare_amount"]).show(3)
df.select(F.col("fare_amount")).show(3)

+-----------+
|fare_amount|
+-----------+
|       17.7|
|       10.0|
|       23.3|
+-----------+
only showing top 3 rows
+-----------+
|fare_amount|
+-----------+
|       17.7|
|       10.0|
|       23.3|
+-----------+
only showing top 3 rows


### filter

Теж три синтаксиси — SQL-рядок, атрибут, `F.col()`.

In [26]:
df.filter(df["fare_amount"] > 50)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-01-01 00:35:16,2024-01-01 01:11:52,2,8.2,1,N,246,190,1,59.0,3.5,0.5,14.15,6.94,1.0,85.09,2.5,0.0
1,2024-01-01 00:42:05,2024-01-01 01:16:49,1,23.9,5,N,263,265,1,120.0,0.0,0.0,0.0,6.94,1.0,127.94,0.0,0.0
2,2024-01-01 00:46:16,2024-01-01 01:20:10,1,20.85,2,N,132,239,1,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:54:16,2024-01-01 01:27:40,1,13.74,1,N,239,95,1,56.9,1.0,0.5,13.77,6.94,1.0,82.61,2.5,0.0
2,2024-01-01 00:50:28,2024-01-01 01:38:39,1,20.34,1,N,132,26,1,80.0,1.0,0.5,2.0,0.0,1.0,86.25,0.0,1.75
2,2024-01-01 00:06:29,2024-01-01 00:31:26,2,16.4,2,N,132,170,2,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:16:21,2024-01-01 00:46:36,1,23.0,4,N,70,265,2,125.5,6.0,0.5,0.0,0.0,1.0,134.75,0.0,1.75
2,2024-01-01 00:55:31,2024-01-01 01:21:15,1,18.42,1,N,132,213,2,68.1,1.0,0.5,0.0,6.94,1.0,79.29,0.0,1.75
2,2024-01-01 00:33:08,2024-01-01 00:59:55,2,20.59,2,N,132,141,1,70.0,0.0,0.5,20.67,6.94,1.0,103.36,2.5,1.75
1,2024-01-01 00:38:11,2024-01-01 01:39:04,1,6.9,1,N,231,24,1,54.1,3.5,0.5,8.87,0.0,1.0,67.97,2.5,0.0


In [27]:
df.filter("fare_amount > 50")

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-01-01 00:35:16,2024-01-01 01:11:52,2,8.2,1,N,246,190,1,59.0,3.5,0.5,14.15,6.94,1.0,85.09,2.5,0.0
1,2024-01-01 00:42:05,2024-01-01 01:16:49,1,23.9,5,N,263,265,1,120.0,0.0,0.0,0.0,6.94,1.0,127.94,0.0,0.0
2,2024-01-01 00:46:16,2024-01-01 01:20:10,1,20.85,2,N,132,239,1,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:54:16,2024-01-01 01:27:40,1,13.74,1,N,239,95,1,56.9,1.0,0.5,13.77,6.94,1.0,82.61,2.5,0.0
2,2024-01-01 00:50:28,2024-01-01 01:38:39,1,20.34,1,N,132,26,1,80.0,1.0,0.5,2.0,0.0,1.0,86.25,0.0,1.75
2,2024-01-01 00:06:29,2024-01-01 00:31:26,2,16.4,2,N,132,170,2,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:16:21,2024-01-01 00:46:36,1,23.0,4,N,70,265,2,125.5,6.0,0.5,0.0,0.0,1.0,134.75,0.0,1.75
2,2024-01-01 00:55:31,2024-01-01 01:21:15,1,18.42,1,N,132,213,2,68.1,1.0,0.5,0.0,6.94,1.0,79.29,0.0,1.75
2,2024-01-01 00:33:08,2024-01-01 00:59:55,2,20.59,2,N,132,141,1,70.0,0.0,0.5,20.67,6.94,1.0,103.36,2.5,1.75
1,2024-01-01 00:38:11,2024-01-01 01:39:04,1,6.9,1,N,231,24,1,54.1,3.5,0.5,8.87,0.0,1.0,67.97,2.5,0.0


In [28]:
df.filter(F.col("fare_amount") > 50)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-01-01 00:35:16,2024-01-01 01:11:52,2,8.2,1,N,246,190,1,59.0,3.5,0.5,14.15,6.94,1.0,85.09,2.5,0.0
1,2024-01-01 00:42:05,2024-01-01 01:16:49,1,23.9,5,N,263,265,1,120.0,0.0,0.0,0.0,6.94,1.0,127.94,0.0,0.0
2,2024-01-01 00:46:16,2024-01-01 01:20:10,1,20.85,2,N,132,239,1,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:54:16,2024-01-01 01:27:40,1,13.74,1,N,239,95,1,56.9,1.0,0.5,13.77,6.94,1.0,82.61,2.5,0.0
2,2024-01-01 00:50:28,2024-01-01 01:38:39,1,20.34,1,N,132,26,1,80.0,1.0,0.5,2.0,0.0,1.0,86.25,0.0,1.75
2,2024-01-01 00:06:29,2024-01-01 00:31:26,2,16.4,2,N,132,170,2,70.0,0.0,0.5,0.0,6.94,1.0,82.69,2.5,1.75
2,2024-01-01 00:16:21,2024-01-01 00:46:36,1,23.0,4,N,70,265,2,125.5,6.0,0.5,0.0,0.0,1.0,134.75,0.0,1.75
2,2024-01-01 00:55:31,2024-01-01 01:21:15,1,18.42,1,N,132,213,2,68.1,1.0,0.5,0.0,6.94,1.0,79.29,0.0,1.75
2,2024-01-01 00:33:08,2024-01-01 00:59:55,2,20.59,2,N,132,141,1,70.0,0.0,0.5,20.67,6.94,1.0,103.36,2.5,1.75
1,2024-01-01 00:38:11,2024-01-01 01:39:04,1,6.9,1,N,231,24,1,54.1,3.5,0.5,8.87,0.0,1.0,67.97,2.5,0.0


In [29]:
df.filter((F.col("fare_amount") > 0) & (F.col("passenger_count") > 0))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0
2,2024-01-01 00:35:22,2024-01-01 00:41:41,2,0.75,1,N,107,137,1,7.9,1.0,0.5,0.0,0.0,1.0,12.9,2.5,0.0


### withColumn — додавання та трансформація колонок

In [30]:
df.withColumn("fare_plus_tip", F.col("fare_amount") + F.col("tip_amount"))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,fare_plus_tip
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,17.7
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,13.75
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,26.3
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,12.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,11.100000000000001
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,36.5
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,55.7
1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,25.4
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,31.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,3.0


In [31]:
df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,pickup_hour
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,0
1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,0
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,0


In [32]:
df.withColumn(
    "trip_duration_min",
    F.timestamp_diff("SECOND", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")) / 60,
)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_min
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,19.8
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,6.6
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,17.916666666666668
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,8.3
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,6.1
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,32.38333333333333
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,26.05
1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,28.0
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,28.183333333333334
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,1.1333333333333333


### transform — кастомна трансформація як функція

In [33]:
def add_duration(d):
    return d.withColumn(
        "trip_duration_min",
        F.timestamp_diff("SECOND", F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")) / 60,
    )

df.transform(add_duration).select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_min")

tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min
2024-01-01 00:57:55,2024-01-01 01:17:43,19.8
2024-01-01 00:03:00,2024-01-01 00:09:36,6.6
2024-01-01 00:17:06,2024-01-01 00:35:01,17.916666666666668
2024-01-01 00:36:38,2024-01-01 00:44:56,8.3
2024-01-01 00:46:51,2024-01-01 00:52:57,6.1
2024-01-01 00:54:08,2024-01-01 01:26:31,32.38333333333333
2024-01-01 00:49:44,2024-01-01 01:15:47,26.05
2024-01-01 00:30:40,2024-01-01 00:58:40,28.0
2024-01-01 00:26:01,2024-01-01 00:54:12,28.183333333333334
2024-01-01 00:28:08,2024-01-01 00:29:16,1.1333333333333333


### alias, withColumnRenamed, drop

In [34]:
df.select(
    F.col("fare_amount").alias("fare_usd"),
    F.col("trip_distance").alias("distance_mi"),
)

fare_usd,distance_mi
17.7,1.72
10.0,1.8
23.3,4.7
10.0,1.4
7.9,0.8
29.6,4.7
45.7,10.82
25.4,3.0
31.0,5.44
3.0,0.04


In [35]:
df.withColumnRenamed("fare_amount", "fare_usd").printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_usd: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [36]:
df.drop("VendorID", "store_and_fwd_flag").printSchema()

root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



### orderBy / sort

In [37]:
df.orderBy("fare_amount")

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-22 16:40:43,2024-01-22 19:11:20,1,157.25,5,N,265,265,2,-899.0,0.0,0.0,0.0,0.0,-1.0,-900.0,0.0,0.0
2,2024-01-18 11:20:40,2024-01-18 11:20:56,4,0.0,5,N,33,33,3,-800.0,0.0,0.0,0.0,0.0,-1.0,-801.0,0.0,0.0
2,2024-01-18 11:26:33,2024-01-18 11:26:42,4,0.0,5,N,33,33,3,-800.0,0.0,0.0,0.0,0.0,-1.0,-801.0,0.0,0.0
2,2024-01-26 23:59:09,2024-01-27 02:45:39,1,120.76,4,N,143,265,4,-744.3,-1.0,-0.5,0.0,-6.94,-1.0,-753.74,0.0,0.0
2,2024-01-21 20:33:50,2024-01-21 20:33:56,1,0.0,5,N,265,265,2,-709.0,0.0,0.0,0.0,0.0,-1.0,-710.0,0.0,0.0
2,2024-01-07 12:38:39,2024-01-07 12:38:59,4,0.11,5,N,246,246,3,-700.0,0.0,0.0,20.0,-14.75,-1.0,-695.75,0.0,0.0
2,2024-01-24 15:21:37,2024-01-24 15:21:49,1,0.0,5,N,66,66,3,-670.0,0.0,0.0,0.0,0.0,-1.0,-671.0,0.0,0.0
2,2024-01-21 18:08:49,2024-01-21 20:15:48,1,110.46,4,N,132,265,2,-669.4,0.0,0.0,141.49,-37.06,-1.0,-567.72,0.0,-1.75
2,2024-01-29 19:19:21,2024-01-29 19:24:05,1,2.46,5,N,132,219,4,-650.0,0.0,0.0,0.0,0.0,-1.0,-652.75,0.0,-1.75
2,2024-01-28 14:41:20,2024-01-28 16:37:25,1,92.34,4,N,138,265,4,-607.8,-5.0,0.0,0.0,-22.32,-1.0,-637.87,0.0,-1.75


In [38]:
df.orderBy(F.col("fare_amount").desc())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
1,2024-01-20 11:18:47,2024-01-20 11:18:47,0,0.0,99,N,264,264,3,5000.0,0.0,0.0,0.0,0.0,0.0,5000.0,0.0,0.0
1,2024-01-20 11:19:33,2024-01-20 11:19:33,0,0.0,99,N,264,264,3,5000.0,0.0,0.0,0.0,0.0,0.0,5000.0,0.0,0.0
1,2024-01-20 11:20:15,2024-01-20 11:20:15,0,0.0,99,N,264,264,3,2500.0,0.0,0.0,0.0,0.0,0.0,2500.0,0.0,0.0
1,2024-01-20 11:27:48,2024-01-20 11:27:48,0,0.0,99,N,264,264,1,2500.0,0.0,0.0,0.0,0.0,0.0,2500.0,0.0,0.0
1,2024-01-24 13:44:43,2024-01-24 13:44:43,0,0.0,99,N,264,264,1,2500.0,0.0,0.0,0.0,0.0,0.0,2500.0,0.0,0.0
2,2024-01-14 10:08:11,2024-01-16 13:54:22,1,31.95,1,N,220,220,2,2221.3,0.0,0.5,0.0,0.0,1.0,2225.3,2.5,0.0
2,2024-01-02 07:50:08,2024-01-02 11:29:29,1,233.25,4,N,168,265,2,1616.5,0.0,0.0,0.0,0.0,1.0,1617.5,0.0,0.0
1,2024-01-20 11:22:27,2024-01-20 11:22:27,0,0.0,99,N,264,264,3,1000.0,0.0,0.0,0.0,0.0,0.0,1000.0,0.0,0.0
2,2024-01-06 21:01:38,2024-01-06 23:41:44,1,142.62,4,N,132,265,2,912.3,1.0,0.0,0.0,26.63,1.0,940.93,0.0,0.0
2,2024-01-22 16:40:43,2024-01-22 19:11:20,1,157.25,5,N,265,265,2,899.0,0.0,0.0,0.0,0.0,1.0,900.0,0.0,0.0


In [39]:
df.sort(df["trip_distance"].desc())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-30 06:37:00,2024-01-30 06:50:00,NULL,312722.3,NULL,NULL,151,162,0,14.46,0.0,0.5,3.69,0.0,1.0,22.15,NULL,NULL
2,2024-01-25 08:39:00,2024-01-25 09:06:00,NULL,97793.92,NULL,NULL,4,13,0,29.71,0.0,0.5,3.3,0.0,0.3,36.31,NULL,NULL
2,2024-01-20 08:01:00,2024-01-20 08:26:00,NULL,82015.45,NULL,NULL,40,43,0,16.56,0.0,0.5,1.0,0.0,1.0,21.56,NULL,NULL
2,2024-01-21 11:58:00,2024-01-21 12:12:00,NULL,72975.97,NULL,NULL,209,211,0,12.7,0.0,0.5,3.34,0.0,1.0,20.04,NULL,NULL
2,2024-01-17 08:41:00,2024-01-17 09:23:00,NULL,71752.26,NULL,NULL,33,161,0,41.06,0.0,0.5,4.51,0.0,1.0,49.57,NULL,NULL
2,2024-01-05 15:46:00,2024-01-05 16:17:00,NULL,59282.45,NULL,NULL,74,47,0,32.02,0.0,0.5,0.0,0.0,1.0,33.52,NULL,NULL
2,2024-01-09 07:13:00,2024-01-09 07:17:00,NULL,59076.43,NULL,NULL,141,162,0,13.82,0.0,0.5,5.35,0.0,1.0,23.17,NULL,NULL
2,2024-01-24 12:26:00,2024-01-24 12:42:00,NULL,58298.51,NULL,NULL,151,236,0,12.94,0.0,0.5,1.69,0.0,1.0,18.63,NULL,NULL
2,2024-01-12 05:16:00,2024-01-12 05:26:00,NULL,51619.36,NULL,NULL,238,186,0,16.17,0.0,0.5,4.03,0.0,1.0,24.2,NULL,NULL
2,2024-01-12 07:18:00,2024-01-12 07:38:00,NULL,44018.64,NULL,NULL,233,138,0,32.75,0.0,0.5,8.74,6.94,1.0,52.43,NULL,NULL


### limit / sample / randomSplit

In [40]:
df.limit(3)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0


In [41]:
ic(df.sample(fraction=0.01, seed=42).count())

ic| df.sample(fraction=0.01, seed=42).count(): 29893


29893

In [42]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
ic(train.count(), test.count())

ic| train.count(): 2371338, test.count(): 593286


(2371338, 593286)

### describe та summary

In [43]:
df.select("fare_amount", "trip_distance", "passenger_count").describe()

summary,fare_amount,trip_distance,passenger_count
count,2964624,2964624,2824462
mean,18.175061916791037,3.6521691789583146,1.3392808966805005
stddev,18.94954770590526,225.46257238220082,0.8502816924800847
min,-899.0,0.0,0
max,5000.0,312722.3,9


In [44]:
df.select("fare_amount", "trip_distance").summary("count", "min", "25%", "50%", "75%", "max")

summary,fare_amount,trip_distance
count,2964624,2964624
min,-899.0,0.0
25%,8.6,1.0
50%,12.8,1.68
75%,20.5,3.11
max,5000.0,312722.3


### toPandas — конвертація для downstream-аналізу

In [45]:
df.select("PULocationID", "fare_amount").limit(1000).toPandas().head(3)

,PULocationID,fare_amount
0,186,17.7
1,140,10.0
2,236,23.3


## 5. Агрегації

`groupBy` повертає `GroupedData` — **transformation** (lazy).
Агрегаційна функція (`count`, `avg`, `agg`) — **action**, запускає обчислення.

In [46]:
df.groupBy("payment_type").count().orderBy("payment_type")

payment_type,count
0,140162
1,2319046
2,439191
3,19597
4,46628


In [47]:
df.groupBy("payment_type").avg("fare_amount").orderBy("payment_type")

payment_type,avg(fare_amount)
0,20.016193904200065
1,18.557432202724847
2,17.866037304954105
3,6.752568760524564
4,1.3348886934888922


### agg — кілька агрегацій одночасно

In [48]:
df.groupBy("PULocationID").agg(
    F.count("*").alias("trips"),
    F.avg("fare_amount").alias("avg_fare"),
    F.max("trip_distance").alias("max_distance"),
).orderBy(F.col("trips").desc())

PULocationID,trips,avg_fare,max_distance
132,145240,59.403120627926164,10879.28
161,143471,15.206993329662337,38202.66
237,142708,12.176191944389887,971.8
236,136465,12.71395566628795,58.81
162,106717,14.785921268401518,71.18
230,106324,17.54252972047708,80.0
186,104523,15.786554920926461,94.0
142,104080,13.427459454265971,58.78
138,89533,41.46328426390264,176.43
239,88474,13.44872957026913,15015.12


### Глобальна агрегація (без groupBy)

In [49]:
df.agg(
    F.avg("fare_amount").alias("avg_fare"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.countDistinct("PULocationID").alias("distinct_zones"),
)

avg_fare,min_fare,max_fare,distinct_zones
18.17506191678942,-899.0,5000.0,260


### rollup

Ієрархічна агрегація: grand total + subtotals.
Рядок з `NULL` у `VendorID` — grand total по всіх вендорах.

In [50]:
df.rollup("VendorID", "payment_type").agg(
    F.count("*").alias("trips"),
    F.sum("fare_amount").alias("total_fare"),
).orderBy("VendorID", "payment_type")

VendorID,payment_type,trips,total_fare
NULL,NULL,2964624,5.3882224760004714E7
1,NULL,729732,1.287214654000019E7
1,0,48455,899224.6199999736
1,1,564618,1.0048705309999956E7
1,2,104722,1729477.62999999
1,3,8120,132345.78999999986
1,4,3817,62393.19000000005
2,NULL,2234632,4.099788439000192E7
2,0,91447,1894091.3200000597
2,1,1754428,3.2986833609999E7


### approxQuantile, corr, cov

In [51]:
ic(df.approxQuantile("fare_amount", [0.25, 0.5, 0.75], 0.01))

ic| df.approxQuantile("fare_amount", [0.25, 0.5, 0.75], 0.01): [8.6, 12.8, 20.5]


[8.6, 12.8, 20.5]

In [52]:
ic(df.corr("fare_amount", "trip_distance"))

ic| df.corr("fare_amount", "trip_distance"): 0.016064980070449663


0.016064980070449663

In [53]:
ic(df.cov("fare_amount", "trip_distance"))

ic| df.cov("fare_amount", "trip_distance"): 68.63624208788823


68.63624208788823

## 6. Joins

Той самий синтаксис `join(other, on=..., how=...)`, що й у Pandas.
Spark вибирає join strategy автоматично — **broadcast**, **sort-merge**, або **hash**.
`explain()` покаже яку саме.

In [54]:
zones = (
    spark.read.parquet(ZONE_PARQUET)
    .select(
        F.col("LocationID").alias("PULocationID"),
        F.col("Zone").alias("PU_Zone"),
        F.col("Borough").alias("PU_Borough"),
    )
)
zones

PULocationID,PU_Zone,PU_Borough
1,Newark Airport,EWR
2,Jamaica Bay,Queens
3,Allerton/Pelham G...,Bronx
4,Alphabet City,Manhattan
5,Arden Heights,Staten Island
6,Arrochar/Fort Wad...,Staten Island
7,Astoria,Queens
8,Astoria Park,Queens
9,Auburndale,Queens
10,Baisley Park,Queens


### inner join

In [55]:
df_joined = df.join(zones, on="PULocationID", how="inner")
df_joined.select("PULocationID", "PU_Zone", "PU_Borough", "fare_amount")

PULocationID,PU_Zone,PU_Borough,fare_amount
186,Penn Station/Madi...,Manhattan,17.7
140,Lenox Hill East,Manhattan,10.0
236,Upper East Side N...,Manhattan,23.3
79,East Village,Manhattan,10.0
211,SoHo,Manhattan,7.9
148,Lower East Side,Manhattan,29.6
138,LaGuardia Airport,Queens,45.7
246,West Chelsea/Huds...,Manhattan,25.4
161,Midtown Center,Manhattan,31.0
113,Greenwich Village...,Manhattan,3.0


### left join

In [56]:
df_left = df.join(zones, on="PULocationID", how="left")
ic(df_left.count(), df.count())

ic| df_left.count(): 2964624, df.count(): 2964624


(2964624, 2964624)

### cross join (ілюстрація на малому прикладі)

In [57]:
vendors = spark.createDataFrame(
    [(1, "Creative Mobile"), (2, "VeriFone")],
    ["VendorID", "vendor_name"],
)
payment_types = spark.createDataFrame(
    [(1, "Credit card"), (2, "Cash"), (3, "No charge"), (4, "Dispute"), (5, "Unknown")],
    ["payment_type", "payment_name"],
)
vendors.crossJoin(payment_types)

VendorID,vendor_name,payment_type,payment_name
1,Creative Mobile,1,Credit card
1,Creative Mobile,2,Cash
1,Creative Mobile,3,No charge
1,Creative Mobile,4,Dispute
1,Creative Mobile,5,Unknown
2,VeriFone,1,Credit card
2,VeriFone,2,Cash
2,VeriFone,3,No charge
2,VeriFone,4,Dispute
2,VeriFone,5,Unknown


### explain() — join strategy

Zones має 265 рядків — Spark вибирає `BroadcastHashJoin` (весь датасет відправляється
на кожен executor, уникаючи shuffle).

In [58]:
df.join(zones, on="PULocationID", how="inner").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [PULocationID#7, VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, PU_Zone#2891, PU_Borough#2892]
   +- BroadcastHashJoin [cast(PULocationID#7 as bigint)], [PULocationID#2890L], Inner, BuildRight, false
      :- Filter isnotnull(PULocationID#7)
      :  +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] Batched: true, DataFilters: [isnotnull(PULocationID#7)], Format: P

## 7. Операції "з теорії множин"

In [59]:
first_hour = (
    df.filter(F.hour("tpep_pickup_datetime") == 0)
    .select("PULocationID", "DOLocationID", "fare_amount")
)
second_hour = (
    df.filter(F.hour("tpep_pickup_datetime") == 1)
    .select("PULocationID", "DOLocationID", "fare_amount")
)
ic(first_hour.count(), second_hour.count())

ic| first_hour.count(): 79094, second_hour.count(): 53627


(79094, 53627)

### union

In [60]:
ic(first_hour.union(second_hour).count())

ic| first_hour.union(second_hour).count(): 132721


132721

### intersect

Рядки, що зустрічаються в **обох** DataFrame (з однаковими значеннями у всіх колонках).

In [61]:
ic(first_hour.intersect(second_hour).count())

ic| first_hour.intersect(second_hour).count(): 12173


12173

### exceptAll

In [62]:
ic(first_hour.exceptAll(second_hour).count())

ic| first_hour.exceptAll(second_hour).count(): 49287


49287

## 8. Null-handling

In [63]:
df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]
)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,0,0,140162,0,140162,140162,0,0,0,0,0,0,0,0,0,0,140162,140162


In [64]:
df.filter(F.col("passenger_count").isNull())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:34:19,2024-01-01 00:51:22,NULL,2.04,NULL,NULL,143,141,0,12.72,0.0,0.5,0.0,0.0,1.0,16.72,NULL,NULL
1,2024-01-01 00:14:31,2024-01-01 00:19:29,NULL,1.6,NULL,NULL,236,238,0,9.3,1.0,0.5,2.86,0.0,1.0,17.16,NULL,NULL
1,2024-01-01 00:35:11,2024-01-01 01:13:40,NULL,0.0,NULL,NULL,142,79,0,21.01,0.0,0.5,0.0,0.0,1.0,25.01,NULL,NULL
1,2024-01-01 00:33:37,2024-01-01 00:50:34,NULL,0.0,NULL,NULL,237,4,0,17.79,0.0,0.5,0.0,0.0,1.0,21.79,NULL,NULL
1,2024-01-01 00:49:04,2024-01-01 01:01:16,NULL,0.0,NULL,NULL,244,50,0,34.65,0.0,0.5,0.0,0.0,1.0,38.65,NULL,NULL
2,2024-01-01 00:47:00,2024-01-01 01:11:05,NULL,4.58,NULL,NULL,202,83,0,28.14,0.0,0.5,0.0,0.0,1.0,29.64,NULL,NULL
2,2024-01-01 00:48:10,2024-01-01 01:00:24,NULL,1.47,NULL,NULL,224,144,0,12.82,0.0,0.5,0.0,0.0,1.0,16.82,NULL,NULL
1,2024-01-01 00:53:57,2024-01-01 01:09:58,NULL,1.7,NULL,NULL,79,249,0,10.16,0.0,0.5,0.0,0.0,1.0,11.66,NULL,NULL
2,2024-01-01 00:55:27,2024-01-01 01:38:45,NULL,4.8,NULL,NULL,158,239,0,33.12,0.0,0.5,7.42,0.0,1.0,44.54,NULL,NULL
2,2024-01-01 00:36:00,2024-01-01 00:43:00,NULL,1.27,NULL,NULL,211,90,0,11.74,0.0,0.5,3.15,0.0,1.0,18.89,NULL,NULL


In [65]:
df.filter(F.col("passenger_count").isNotNull())

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,246,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0


### fillna / replace

In [66]:
df.fillna({"passenger_count": 1.0}).select("passenger_count").summary("count", "min")

summary,passenger_count
count,2964624
min,0


In [67]:
df.replace(0, 1, subset="VendorID").groupBy("VendorID").count()

VendorID,count
1,729732
2,2234632
6,260


### dropna / dropDuplicates / distinct

In [68]:
ic(df.count(), df.dropna().count())

ic| df.count(): 2964624, df.dropna().count(): 2824462


(2964624, 2824462)

In [69]:
ic(df.count(), df.distinct().count())

ic| df.count(): 2964624, df.distinct().count(): 2964624


(2964624, 2964624)

In [70]:
df.dropDuplicates(["PULocationID", "DOLocationID", "tpep_pickup_datetime"])

26/08/13 18:40:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 06:57:44,2024-01-01 06:57:56,4,0.0,3,N,1,1,4,-23.0,0.0,0.0,0.0,0.0,-1.0,-24.0,0.0,0.0
2,2024-01-02 05:13:37,2024-01-02 05:15:00,1,0.0,3,N,1,1,1,23.7,1.0,0.0,99.0,0.0,1.0,124.7,0.0,0.0
2,2024-01-03 18:58:36,2024-01-03 18:58:54,1,0.0,5,N,1,1,1,87.0,0.0,0.0,0.1,0.0,1.0,88.1,0.0,0.0
1,2024-01-05 17:17:38,2024-01-05 17:18:30,1,0.3,5,N,1,1,1,150.0,0.0,0.0,0.0,0.0,1.0,151.0,0.0,0.0
2,2024-01-06 06:50:37,2024-01-06 06:50:43,3,0.0,5,N,1,1,1,110.5,0.0,0.0,22.3,0.0,1.0,133.8,0.0,0.0
2,2024-01-07 18:57:33,2024-01-07 18:57:46,2,0.0,5,N,1,1,1,50.0,0.0,0.0,0.0,0.0,1.0,51.0,0.0,0.0
2,2024-01-08 15:10:29,2024-01-08 15:10:34,2,0.0,5,N,1,1,1,50.0,0.0,0.0,10.2,0.0,1.0,61.2,0.0,0.0
2,2024-01-09 21:03:15,2024-01-09 21:03:19,1,0.0,5,N,1,1,1,79.0,0.0,0.0,0.0,0.0,1.0,80.0,0.0,0.0
1,2024-01-10 14:23:59,2024-01-10 14:25:47,1,0.3,5,N,1,1,1,116.0,0.0,0.0,20.0,0.0,1.0,137.0,0.0,0.0
2,2024-01-11 16:57:44,2024-01-11 16:57:56,1,0.0,5,N,1,1,1,130.0,0.0,0.0,0.0,0.0,1.0,131.0,0.0,0.0


## 9. SparkSQL

**DataFrame API** і **SparkSQL** — два рівноцінні способи написати той самий запит.
Обидва компілюються в ідентичний план виконання через **Catalyst optimizer**.
Вибір — питання стилю та читабельності.

In [71]:
df.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

### spark.sql — SQL-запит

In [72]:
spark.sql("""
    SELECT payment_type, COUNT(*) AS trips, ROUND(AVG(fare_amount), 2) AS avg_fare
    FROM trips
    WHERE fare_amount > 0
    GROUP BY payment_type
    ORDER BY trips DESC
""")

payment_type,trips,avg_fare
1,2318858,18.56
2,430627,18.63
0,138046,20.35
4,25039,19.68
3,13713,17.23


Той самий запит через DataFrame API:

In [73]:
(
    df.filter(F.col("fare_amount") > 0)
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    )
    .orderBy(F.col("trips").desc())
)

payment_type,trips,avg_fare
1,2318858,18.56
2,430627,18.63
0,138046,20.35
4,25039,19.68
3,13713,17.23


### join через SQL

In [74]:
spark.sql("""
    SELECT t.PULocationID, z.PU_Zone, COUNT(*) AS trips
    FROM trips t
    LEFT JOIN zones z ON t.PULocationID = z.PULocationID
    GROUP BY t.PULocationID, z.PU_Zone
    ORDER BY trips DESC
    LIMIT 10
""")

PULocationID,PU_Zone,trips
132,JFK Airport,145240
161,Midtown Center,143471
237,Upper East Side S...,142708
236,Upper East Side N...,136465
162,Midtown East,106717
230,Times Sq/Theatre ...,106324
186,Penn Station/Madi...,104523
142,Lincoln Square East,104080
138,LaGuardia Airport,89533
239,Upper West Side S...,88474


### selectExpr — SQL-вирази всередині DataFrame API

In [75]:
df.selectExpr(
    "fare_amount AS fare_usd",
    "trip_distance AS distance_mi",
    "fare_amount / NULLIF(trip_distance, 0) AS fare_per_mile",
)

fare_usd,distance_mi,fare_per_mile
17.7,1.72,10.290697674418604
10.0,1.8,5.555555555555555
23.3,4.7,4.957446808510638
10.0,1.4,7.142857142857143
7.9,0.8,9.875
29.6,4.7,6.297872340425532
45.7,10.82,4.22365988909427
25.4,3.0,8.466666666666667
31.0,5.44,5.698529411764706
3.0,0.04,75.0


### Metadata

In [76]:
ic(df.inputFiles())

ic| df.inputFiles(): ['file:///Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/lesson-12-spark-structured-data-1/code/data/landing/yellow_tripdata_2024-01.parquet']


['file:///Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/lesson-12-spark-structured-data-1/code/data/landing/yellow_tripdata_2024-01.parquet']

In [77]:
ic(df.isEmpty())

ic| df.isEmpty(): False


False

In [78]:
ic(df.isStreaming)

ic| df.isStreaming: False


False

## 10. Window functions

`Window.partitionBy(...).orderBy(...)` — аналог `PARTITION BY ... ORDER BY` в SQL.
Те саме концептуально, що й у занятті 03 (DuckDB window functions) — але тепер розподілено.

In [79]:
window_spec = Window.partitionBy("PULocationID").orderBy(F.col("fare_amount").desc())

### row_number, rank, dense_rank

Знаходимо топ поїздки з кожної зони відправлення.

In [80]:
df.withColumn("row_num", F.row_number().over(window_spec)).where("row_num = 1").select(
    "PULocationID", "fare_amount", "row_num"
)

PULocationID,fare_amount,row_num
1,267.94,1
2,70.0,1
3,216.5,1
4,276.0,1
6,160.0,1
7,200.0,1
8,150.0,1
9,70.0,1
10,325.0,1
11,73.5,1


In [81]:
df.withColumn("rank", F.rank().over(window_spec)).where("rank = 1").select(
    "PULocationID", "fare_amount", "rank"
)

PULocationID,fare_amount,rank
1,267.94,1
2,70.0,1
3,216.5,1
4,276.0,1
6,160.0,1
7,200.0,1
8,150.0,1
8,150.0,1
9,70.0,1
10,325.0,1


In [82]:
df.withColumn("dense_rank", F.dense_rank().over(window_spec)).where(
    F.col("dense_rank").isin([1, 2])
).select("PULocationID", "fare_amount", "dense_rank")

PULocationID,fare_amount,dense_rank
12,114.3,1
12,92.3,2
22,94.5,1
22,69.5,2
22,69.5,2
26,110.0,1
26,103.0,2
27,82.0,1
27,63.5,2
28,225.0,1


### lag — попереднє значення у вікні

Середній тариф по годині + значення попередньої години.

In [83]:
# REWRITE. Add partition in window spec. You may change the logic of the actual window function if needed.

hour_window = Window.orderBy("pickup_hour")

(
    df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
    .orderBy("pickup_hour")
    .withColumn("prev_hour_fare", F.lag("avg_fare", 1).over(hour_window))
)

26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 18:40:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/08/13 1

pickup_hour,avg_fare,prev_hour_fare
0,19.2,NULL
1,17.53,19.2
2,16.48,17.53
3,18.15,16.48
4,22.52,18.15
5,26.62,22.52
6,21.65,26.62
7,18.54,21.65
8,17.65,18.54
9,17.71,17.65



### String functions (після join із довідником зон)

In [84]:
df_with_zones = df.join(zones, on="PULocationID", how="left")

In [85]:
# like — SQL wildcards
df_with_zones.filter(F.col("PU_Zone").like("%Airport%")).select(
    "PU_Zone", "PU_Borough", "fare_amount"
)

PU_Zone,PU_Borough,fare_amount
LaGuardia Airport,Queens,45.7
JFK Airport,Queens,21.2
LaGuardia Airport,Queens,35.2
JFK Airport,Queens,70.0
JFK Airport,Queens,80.0
JFK Airport,Queens,70.0
JFK Airport,Queens,68.1
JFK Airport,Queens,70.0
LaGuardia Airport,Queens,33.8
JFK Airport,Queens,42.9


In [86]:
# startswith / endswith / contains
df_with_zones.filter(F.col("PU_Borough").startswith("Man")).select(
    "PU_Borough", "PU_Zone"
).distinct()

PU_Borough,PU_Zone
Manhattan,Times Sq/Theatre ...
Manhattan,Central Harlem North
Manhattan,Lenox Hill East
Manhattan,Battery Park City
Manhattan,Upper West Side S...
Manhattan,East Harlem South
Manhattan,Midtown Center
Manhattan,Gramercy
Manhattan,Kips Bay
Manhattan,Marble Hill


In [87]:
df_with_zones.filter(F.col("PU_Zone").contains("Heights")).select("PU_Zone").distinct()

PU_Zone
Dyker Heights
Jackson Heights
Stuyvesant Heights
Crown Heights North
Washington Height...
Brooklyn Heights
Cambria Heights
University Height...
Hamilton Heights
Prospect Heights


In [88]:
df_with_zones.filter(F.col("PU_Zone").endswith("Airport")).select("PU_Zone").distinct()

PU_Zone
Newark Airport
LaGuardia Airport
JFK Airport


## 11. Pandas API on Spark (`pyspark.pandas`)

`pyspark.pandas` — pandas-сумісний API, вбудований у PySpark 3.2+. Той самий синтаксис, що у pandas, але виконується розподілено.

| | `pandas` | `pyspark.pandas` | PySpark DataFrame |
|---|---|---|---|
| Виконання | локально | розподілено | розподілено |
| Синтаксис | pandas | pandas | Spark API |
| Розмір даних | < RAM | будь-який | будь-який |

Конвертація: `df.pandas_api()` → `pyspark.pandas.DataFrame`; `.to_spark()` → назад у Spark DataFrame.

In [89]:
import pyspark.pandas as ps

/Users/illia/projects/de_course_materials/.venv/lib/python3.12/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [90]:
psdf = df.pandas_api()
type(psdf)

pyspark.pandas.frame.DataFrame

### Фільтрація — pandas-синтаксис

In [91]:
psdf[psdf["fare_amount"] > 50][["fare_amount", "trip_distance", "PULocationID"]].head(5)

,fare_amount,trip_distance,PULocationID
12,59.0,8.20,246
24,120.0,23.90,263
75,70.0,20.85,132
95,56.9,13.74,239
98,80.0,20.34,132


### Додавання колонок — `.dt` accessor

In [92]:
psdf["pickup_hour"] = psdf["tpep_pickup_datetime"].dt.hour
psdf[["tpep_pickup_datetime", "pickup_hour"]].head(5)

,tpep_pickup_datetime,pickup_hour
0,2024-01-01 00:57:55,0
1,2024-01-01 00:03:00,0
2,2024-01-01 00:17:06,0
3,2024-01-01 00:36:38,0
4,2024-01-01 00:46:51,0


### groupby + agg — кілька метрик

In [93]:
psdf.groupby("payment_type").agg(
    trips=("fare_amount", "count"),
    avg_fare=("fare_amount", "mean"),
).sort_values("trips", ascending=False)

,trips,avg_fare
payment_type,,
1,2319046,18.557432
2,439191,17.866037
0,140162,20.016194
4,46628,1.334889
3,19597,6.752569


### describe

In [94]:
psdf[["fare_amount", "trip_distance", "passenger_count"]].describe()

,fare_amount,trip_distance,passenger_count
count,2.964624e+06,2.964624e+06,2.824462e+06
mean,1.817506e+01,3.652169e+00,1.339281e+00
std,1.894955e+01,2.254626e+02,8.502817e-01
min,-8.990000e+02,0.000000e+00,0.000000e+00
25%,8.600000e+00,1.000000e+00,1.000000e+00
50%,1.280000e+01,1.680000e+00,1.000000e+00
75%,2.050000e+01,3.110000e+00,1.000000e+00
max,5.000000e+03,3.127223e+05,9.000000e+00


### `.to_spark()` — назад у Spark DataFrame

Після роботи через `pyspark.pandas` можна повернутись до Spark DataFrame для запису або подальших Spark-трансформацій.

In [95]:
psdf[["VendorID", "fare_amount", "pickup_hour"]].to_spark().printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)



/Users/illia/projects/de_course_materials/.venv/lib/python3.12/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


## 12. Кешування та управління пам'яттю

**Проблема без кешу:** кожен action перераховує весь lineage від читання файлу. Якщо DataFrame використовується кілька разів — це зайвий shuffle і IO.

`cache()` = `persist(StorageLevel.MEMORY_AND_DISK)` — зберігає DataFrame після першого action.

| Storage Level | Де | Серіалізація | Коли використовувати |
|---|---|---|---|
| `MEMORY_ONLY` | Java heap | ні | DataFrame вміщується в RAM |
| `MEMORY_AND_DISK` | heap → disk при overflow | ні | за замовчуванням (cache()) |
| `MEMORY_ONLY_SER` | heap | так | менше RAM, більше CPU |
| `DISK_ONLY` | disk | так | RAM обмежена, dataset великий |

**Правило:** кешуй тільки якщо DataFrame використовується **≥ 2 рази** в різних actions.

In [96]:
from pyspark import StorageLevel


In [97]:
df_clean = df.filter((F.col("fare_amount") > 0) & (F.col("passenger_count") > 0))

In [98]:
ic(df_clean.count())  
ic(df_clean.storageLevel)

ic| df_clean.count(): 2756810
ic| df_clean.storageLevel: StorageLevel(False, False, False, False, 1)


StorageLevel(False, False, False, False, 1)

In [99]:
df_clean.cache() 

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0
2,2024-01-01 00:35:22,2024-01-01 00:41:41,2,0.75,1,N,107,137,1,7.9,1.0,0.5,0.0,0.0,1.0,12.9,2.5,0.0


In [100]:
ic(df_clean.count())   
ic(df_clean.storageLevel) 

ic| df_clean.count(): 2756810
ic| df_clean.storageLevel: StorageLevel(True, True, False, True, 1)


StorageLevel(True, True, False, True, 1)

In [101]:
df_clean.unpersist()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0
2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75
2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0
2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0
2,2024-01-01 00:35:22,2024-01-01 00:41:41,2,0.75,1,N,107,137,1,7.9,1.0,0.5,0.0,0.0,1.0,12.9,2.5,0.0


### persist() — явний рівень зберігання

`DISK_ONLY` корисний якщо dataset не вміщується в heap, але повторне читання з диску швидше ніж S3/HDFS.

In [102]:
df_joined

PULocationID,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,PU_Zone,PU_Borough
186,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0,Penn Station/Madi...,Manhattan
140,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,N,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,Lenox Hill East,Manhattan
236,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,N,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,Upper East Side N...,Manhattan
79,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,N,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,East Village,Manhattan
211,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,N,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,SoHo,Manhattan
148,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,N,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,Lower East Side,Manhattan
138,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,181,1,45.7,6.0,0.5,10.0,0.0,1.0,64.95,0.0,1.75,LaGuardia Airport,Queens
246,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.0,1,N,231,2,25.4,3.5,0.5,0.0,0.0,1.0,30.4,2.5,0.0,West Chelsea/Huds...,Manhattan
161,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,261,2,31.0,1.0,0.5,0.0,0.0,1.0,36.0,2.5,0.0,Midtown Center,Manhattan
113,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,2,3.0,1.0,0.5,0.0,0.0,1.0,8.0,2.5,0.0,Greenwich Village...,Manhattan


In [103]:
df_joined_cached = df_joined.persist(StorageLevel.DISK_ONLY_3)
df_joined_cached.count()
ic(df_joined_cached.storageLevel)

26/08/13 18:40:51 WARN RandomBlockReplicationPolicy: Expecting 2 replicas with only 0 peer/s.
26/08/13 18:40:51 WARN RandomBlockReplicationPolicy: Expecting 2 replicas with only 0 peer/s.
26/08/13 18:40:51 WARN RandomBlockReplicationPolicy: Expecting 2 replicas with only 0 peer/s.
26/08/13 18:40:51 WARN BlockManager: Block rdd_945_9 replicated to only 0 peer(s) instead of 2 peers
26/08/13 18:40:51 WARN RandomBlockReplicationPolicy: Expecting 2 replicas with only 0 peer/s.
26/08/13 18:40:51 WARN BlockManager: Block rdd_945_0 replicated to only 0 peer(s) instead of 2 peers
26/08/13 18:40:51 WARN BlockManager: Block rdd_945_3 replicated to only 0 peer(s) instead of 2 peers
26/08/13 18:40:51 WARN BlockManager: Block rdd_945_6 replicated to only 0 peer(s) instead of 2 peers
26/08/13 18:40:51 WARN RandomBlockReplicationPolicy: Expecting 2 replicas with only 0 peer/s.
26/08/13 18:40:51 WARN BlockManager: Block rdd_945_2 replicated to only 0 peer(s) instead of 2 peers
26/08/13 18:40:51 WARN Ra

StorageLevel(True, False, False, False, 3)

### unpersist() — звільнення пам'яті

In [104]:
df_clean.unpersist()
df_joined_cached.unpersist()
ic(df_clean.storageLevel)  # StorageLevel(False, False, False, False, 1) — не в кеші

ic| df_clean.storageLevel: StorageLevel(False, False, False, False, 1)


StorageLevel(False, False, False, False, 1)

## 13. Repartition і Coalesce

Кількість партицій визначає рівень паралелізму і кількість вихідних файлів при записі.

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Рух даних між executor'ами | завжди (по мережі) | мінімальний; лише remote fetch, якщо партиції не co-located |
| Розподіл | рівномірний | може бути нерівний |
| Напрямок | ↑ або ↓ | тільки ↓ |
| Застосування | збільшити паралелізм, вирівняти | зменшити перед записом |

> **Важливо:** «без shuffle» ≠ «без руху даних». `coalesce` не додає `Exchange`/stage у план, але коли `DefaultPartitionCoalescer` зливає партиції з різних executor'ів, `BlockManager` дочитує блоки по мережі. Це **data retrieval у межах того самого stage**, а не shuffle. `repartition` навпаки — завжди повний shuffle з `Exchange`.

**Дефолт:** Spark читає Parquet з однієї партиції на файл; великі файли розбиваються за `spark.sql.files.maxPartitionBytes` (128 MB).

In [105]:
ic(df.rdd.getNumPartitions())  # залежить від розміру файлу і maxPartitionBytes

ic| df.rdd.getNumPartitions(): 10


10

### repartition(n) — shuffle, рівномірний розподіл

In [106]:
df_repartitioned = df.repartition(8)
ic(df_repartitioned.rdd.getNumPartitions())

ic| df_repartitioned.rdd.getNumPartitions(): 8


8

In [107]:
df.repartition(8).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=7050]
   +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/lesson-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [108]:
df_coalesced = df.coalesce(2)
ic(df_coalesced.rdd.getNumPartitions())

ic| df_coalesced.rdd.getNumPartitions(): 2


2

In [109]:
df.coalesce(2).explain()

== Physical Plan ==
Coalesce 2
+- *(1) ColumnarToRow
   +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/lesson-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




### repartition by column — hash partitioning

`repartition("col")` розподіляє рядки за hash значення колонки — усі рядки з однаковим `VendorID` потрапляють в одну партицію. Оптимально перед `groupBy` або `join` по тій самій колонці.

In [110]:
df_by_vendor = df.repartition(4, "VendorID")
ic(df_by_vendor.rdd.getNumPartitions())

ic| df_by_vendor.rdd.getNumPartitions(): 4


4

In [111]:
df_by_vendor.groupBy("VendorID").count().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[VendorID#0], functions=[count(1)])
   +- HashAggregate(keys=[VendorID#0], functions=[partial_count(1)])
      +- Exchange hashpartitioning(VendorID#0, 4), REPARTITION_BY_NUM, [plan_id=7114]
         +- FileScan parquet [VendorID#0] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/lesson-..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int>




## 14. Запис даних

In [112]:
(
    df.filter(F.col("fare_amount") > 0)
    .write.mode("overwrite")
    .parquet("data/output/trips_clean.parquet")
)

In [113]:
(
    df_joined.filter(F.col("fare_amount") > 0)
    .write.mode("overwrite")
    .parquet("data/output/trips_with_zones.parquet")
)

### Вплив партиціонування на кількість вихідних файлів

Кількість **непорожніх партицій** DataFrame = кількість файлів при записі (один файл на партицію).
`coalesce(1)` → 1 файл (зручно для невеликих таблиць; антипатерн для великих).
`partitionBy("col")` → директорія `col=value/` на кожне унікальне значення (Hive-style). Усередині кожної директорії — до одного файлу на кожну партицію DataFrame, що містить рядки цього значення.

In [114]:
ic(df.rdd.getNumPartitions())  # 10 — Spark б'є файл на split-и (за розміром, maxPartitionBytes і кількістю ядер), не один-до-одного з файлом


ic| df.rdd.getNumPartitions(): 10


10

In [115]:
# coalesce(1): всі дані в один файл (не потребує shuffle)
(
    df.filter(F.col("fare_amount") > 0)
    .coalesce(1)
    .write.mode("overwrite")
    .parquet("data/output/trips_single_file.parquet")
)

In [116]:
# partitionBy: Hive-style directory partitioning за VendorID
# → директорія на КОЖНЕ значення: VendorID=1/, VendorID=2/, VendorID=6/
# усередині кожної — кілька файлів (по одному на партицію df, що має рядки цього вендора): 3, 3, 1
(
    df.filter(F.col("fare_amount") > 0)
    .write.mode("overwrite")
    .partitionBy("VendorID")
    .parquet("data/output/trips_by_vendor.parquet")
)

In [117]:
# repartition по ТІЙ САМІЙ колонці, що й partitionBy: усі рядки одного VendorID
# хешуються в одну партицію → РІВНО 1 файл на директорію VendorID=*/.
# Це канонічний фікс проблеми "багато дрібних файлів", а не спосіб отримати N файлів на директорію.
(
    df.filter(F.col("fare_amount") > 0)
    .repartition("VendorID")
    .write.mode("overwrite")
    .partitionBy("VendorID")
    .parquet("data/output/trips_by_vendor_1file.parquet")
)

In [118]:
#spark.stop()